In [3]:
library(vegan)
library(tidyverse)
library(pairwiseAdonis)
library(emmeans)

In [4]:
data <- read.csv('../data/descriptors_v2.csv')

### Helper

In [5]:
descriptors <- c("centroid", "logattacktime")
factors <- c("material", "note", "method")

In [6]:
set.seed(20260601)

In [7]:
printHeader <- function(text, width = 70) {
  cat("\n", rep("=", width), "\n", sep = "")
  cat(text, "\n")
  cat(rep("=", width), "\n\n", sep = "")
}

In [8]:
prepData <- function(df, descriptors) {
  standard_data <- df %>%
    select(all_of(descriptors)) %>%
    scale() %>%
    as.data.frame()
  
  return(standard_data)
}

# PERMANOVA

In [9]:
calcDistanceMatrix <- function(data, method = "euclidean") {
  if (is.data.frame(data) && ncol(data) > 1) {
    return(vegdist(data, method = method))
  }
  return(dist(data))
}

In [10]:
execPermanova <- function(dist_matrix, data, factors, permutations, by) {
  terms <- paste(factors, collapse = " * ")
  formula <- as.formula(paste("dist_matrix ~", terms))
  
  model <- adonis2(
    formula,
    data = data,
    permutations = permutations,
    by = by
  )
  
  return(model)
}

In [11]:
univariateAnalysis <- function(df, descriptor, factors, permutations, by) {
  printHeader(paste(toupper(descriptor)))
  
  data <- data.frame(
    descriptor_value = scale(df[[descriptor]]),
    df[factors]
  )

  dist_matrix <- calcDistanceMatrix(data$descriptor_value)
  model <- execPermanova(dist_matrix, data, factors, permutations, by = by)
  
  print(model)
}


In [12]:
multivariateAnalysis <- function(df, descriptors, factors, permutations, by) {
  title <- paste(paste(toupper(descriptors), collapse = " + "))
  printHeader(title)
  
  combined_data <- prepData(df, descriptors)
  dist_matrix <- calcDistanceMatrix(combined_data, "euclidean")
  model <- execPermanova(dist_matrix, df, factors, permutations, by = by)
  
  print(model)
}


In [13]:
PERMANOVA <- function(df, descriptors, factors, permutations = 9999, by) {

  # individual analyses
  for (descriptor in descriptors) {
    model <- univariateAnalysis(df, descriptor, factors, permutations, by = by)
  }
  # combined analysis
  combined_model <- multivariateAnalysis(df, descriptors, factors, permutations, by = by)
}


## Univariate and multivariate

In [14]:
PERMANOVA(
  df = data,
  descriptors = descriptors,
  factors = factors,
  permutations = 9999,
  by = 'term'
)


CENTROID 

Permutation test for adonis under reduced model
Terms added sequentially (first to last)
Permutation: free
Number of permutations: 9999

adonis2(formula = formula, data = data, permutations = permutations, by = by)
                     Df SumOfSqs      R2          F Pr(>F)    
material              1    0.023 0.00029     3.9981 0.0486 *  
note                  3    2.950 0.03734   170.0408 0.0001 ***
method                1   70.155 0.88804 12133.3257 0.0001 ***
material:note         3    0.071 0.00090     4.1127 0.0114 *  
material:method       1    0.003 0.00004     0.5010 0.4888    
note:method           3    5.350 0.06772   308.4416 0.0001 ***
material:note:method  3    0.078 0.00098     4.4816 0.0072 ** 
Residual             64    0.370 0.00468                      
Total                79   79.000 1.00000                      
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

LOGATTACKTIME 

Permutation test for adonis under reduced model
Terms added

# Global

In [15]:
PERMANOVA(
  df = data,
  descriptors = descriptors,
  factors = factors,
  permutations = 9999,
  by = NULL
)


CENTROID 

Permutation test for adonis under reduced model
Permutation: free
Number of permutations: 9999

adonis2(formula = formula, data = data, permutations = permutations, by = by)
         Df SumOfSqs      R2     F Pr(>F)    
Model    15    78.63 0.99532 906.6  1e-04 ***
Residual 64     0.37 0.00468                 
Total    79    79.00 1.00000                 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

LOGATTACKTIME 

Permutation test for adonis under reduced model
Permutation: free
Number of permutations: 9999

adonis2(formula = formula, data = data, permutations = permutations, by = by)
         Df SumOfSqs      R2     F Pr(>F)    
Model    15   56.358 0.71339 10.62  1e-04 ***
Residual 64   22.642 0.28661                 
Total    79   79.000 1.00000                 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

CENTROID + LOGATTACKTIME 

Permutation test for adonis under reduced model
Permutation: free
Number of permutations: 999

# PERMDISP

In [16]:
PERMDISP <- function(df, descriptors, factor, permutations = 9999) {
  
  data <- prepData(df, descriptors)
  dist_matrix <- calcDistanceMatrix(data, "euclidean")
  
  bd <- betadisper(dist_matrix, group = df[[factor]])
  
  # tests
  printHeader(paste(toupper('PARAMETRIC ANOVA')))
  anova_disp <- anova(bd)
  print(anova_disp)

  printHeader(paste(toupper('PERMUTATION TEST')))
  perm_disp <- permutest(bd, permutations = permutations)
  print(perm_disp)

  printHeader(paste(toupper('TUKEY HSD')))
  tuk_disp <- TukeyHSD(bd)
  print(tuk_disp)
}


In [17]:
PERMDISP(
  df = data,
  descriptors = descriptors,
  factor = "method",
  permutations = 9999
)


PARAMETRIC ANOVA 

Analysis of Variance Table

Response: Distances
          Df  Sum Sq Mean Sq F value   Pr(>F)   
Groups     1  3.0447 3.04471  11.634 0.001029 **
Residuals 78 20.4138 0.26171                    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

PERMUTATION TEST 


Permutation test for homogeneity of multivariate dispersions
Permutation: free
Number of permutations: 9999

Response: Distances
          Df  Sum Sq Mean Sq      F N.Perm Pr(>F)    
Groups     1  3.0447 3.04471 11.634   9999  2e-04 ***
Residuals 78 20.4138 0.26171                         
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

TUKEY HSD 

  Tukey multiple comparisons of means
    95% family-wise confidence level

Fit: aov(formula = distances ~ group, data = df)

$group
                  diff      lwr       upr     p adj
natural-bend 0.3901739 0.162435 0.6179128 0.0010286



In [18]:
PERMDISP(
  df = data,
  descriptors = descriptors,
  factor = "material",
  permutations = 9999
)


PARAMETRIC ANOVA 

Analysis of Variance Table

Response: Distances
          Df  Sum Sq Mean Sq F value Pr(>F)
Groups     1  0.2511 0.25106  0.7566 0.3871
Residuals 78 25.8818 0.33182               

PERMUTATION TEST 


Permutation test for homogeneity of multivariate dispersions
Permutation: free
Number of permutations: 9999

Response: Distances
          Df  Sum Sq Mean Sq      F N.Perm Pr(>F)
Groups     1  0.2511 0.25106 0.7566   9999 0.3909
Residuals 78 25.8818 0.33182                     

TUKEY HSD 

  Tukey multiple comparisons of means
    95% family-wise confidence level

Fit: aov(formula = distances ~ group, data = df)

$group
              diff        lwr       upr     p adj
wood-abs 0.1120413 -0.1443911 0.3684737 0.3870528



In [19]:
PERMDISP(
  df = data,
  descriptors = descriptors,
  factor = "note",
  permutations = 9999
)


PARAMETRIC ANOVA 

Analysis of Variance Table

Response: Distances
          Df  Sum Sq Mean Sq F value    Pr(>F)    
Groups     3  5.3265 1.77551  8.8044 4.415e-05 ***
Residuals 76 15.3262 0.20166                      
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

PERMUTATION TEST 


Permutation test for homogeneity of multivariate dispersions
Permutation: free
Number of permutations: 9999

Response: Distances
          Df  Sum Sq Mean Sq      F N.Perm Pr(>F)    
Groups     3  5.3265 1.77551 8.8044   9999  1e-04 ***
Residuals 76 15.3262 0.20166                         
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

TUKEY HSD 

  Tukey multiple comparisons of means
    95% family-wise confidence level

Fit: aov(formula = distances ~ group, data = df)

$group
            diff         lwr       upr     p adj
E4-A4  0.5435863  0.17056161 0.9166109 0.0014768
F4-A4  0.3009469 -0.07207778 0.6739716 0.1562961
G4-A4  0.6781922  0.30516753 1.0512169 0

## PERMDISP within note and method

In [ ]:
PERMDISPNoteMethod <- function(df, descriptors, permutations = 9999) {

  df$note_method <- interaction(df$note, df$method)
  data <- prepData(df, descriptors)
  dist_matrix <- calcDistanceMatrix(data, "euclidean")

  bd <- betadisper(dist_matrix, group = df$note_method)

  printHeader("MEAN DISTANCE TO GROUP CENTROID: NOTE x METHOD")
  mean_dist <- tapply(bd$distances, df$note_method, mean)
  print(round(mean_dist, 3))

  printHeader("NATURAL - BEND BY NOTE")
  natural <- mean_dist[grep("natural", names(mean_dist))]
  bend <- mean_dist[grep("bend", names(mean_dist))]
  print(round(natural[order(names(natural))] - bend[order(names(bend))], 3))

  printHeader("PERMUTATION TEST (8 GROUPS)")
  print(permutest(bd, permutations = permutations))
}

In [35]:
PERMDISPNoteMethod(
  df = data,
  descriptors = descriptors,
  permutations = 9999
)


MEAN DISTANCE TO GROUP CENTROID: NOTE x METHOD 

   A4.bend    E4.bend    F4.bend    G4.bend A4.natural E4.natural F4.natural 
     0.283      0.185      0.253      0.568      0.308      0.364      0.613 
G4.natural 
     0.502 

NATURAL - BEND BY NOTE 

A4.natural E4.natural F4.natural G4.natural 
     0.024      0.179      0.360     -0.066 

PERMUTATION TEST (8 GROUPS) 


Permutation test for homogeneity of multivariate dispersions
Permutation: free
Number of permutations: 9999

Response: Distances
          Df Sum Sq Mean Sq      F N.Perm Pr(>F)
Groups     7  1.733 0.24757 0.9404   9999 0.4877
Residuals 72 18.955 0.26327                     


### Helper

In [22]:
colSignificance <- function(results, p_column = "adjusted_p") {
  results$Significance <- cut(
    results[[p_column]],
    breaks = c(0, 0.001, 0.01, 0.05, 1),
    labels = c("***", "**", "*", "n.s."),
    include.lowest = TRUE
  )
  results[order(results[[p_column]]), ]
}

# Main post-hoc: pairwise PERMANOVA

In [23]:
execPermanovaPair <- function(dist_matrix, df, factor, pair, permutations) {
  indices <- df[[factor]] %in% pair
  D_pair <- as.dist(as.matrix(dist_matrix)[indices, indices])
  df_pair <- df[indices, ]
  
  formula_str <- paste("D_pair ~", factor)
  
  result <- adonis2(as.formula(formula_str), data = df_pair, 
                       permutations = permutations, by = NULL)
  
  data.frame(
    Pair = paste(pair[1], "vs", pair[2]),
    F = result$F[1],
    R2 = result$R2[1],
    p = result$`Pr(>F)`[1]
  )
}


In [24]:
posthocPermanova <- function(df, descriptors, factor, permutations = 9999) {
  
  printHeader(paste("POST-HOC PAIRWISE:", toupper(factor)))
  
  standardized_data <- prepData(df, descriptors)
  dist_matrix <- calcDistanceMatrix(standardized_data, "euclidean")
  
  combinations <- combn(unique(df[[factor]]), 2, simplify = FALSE)
  results <- do.call(rbind, lapply(combinations, function(pair) {
    execPermanovaPair(dist_matrix, df, factor, pair, permutations)
  }))
  
  results$adjusted_p <- p.adjust(results$p, method = "holm")
  results <- colSignificance(results, "adjusted_p")
  
  print(results)
}

In [25]:
posthocPermanova(
  df = data,
  descriptors = descriptors,
  factor = "method",
  permutations = 9999
)


POST-HOC PAIRWISE: METHOD 

             Pair        F        R2     p adjusted_p Significance
1 natural vs bend 174.7491 0.6913935 1e-04      1e-04          ***


In [26]:
posthocPermanova(
  df = data,
  descriptors = descriptors,
  factor = "material",
  permutations = 9999
)


POST-HOC PAIRWISE: MATERIAL 

         Pair         F         R2      p adjusted_p Significance
1 wood vs abs 0.1919095 0.00245434 0.7859     0.7859         n.s.


In [27]:
posthocPermanova(
  df = data,
  descriptors = descriptors,
  factor = "note",
  permutations = 9999
)


POST-HOC PAIRWISE: NOTE 

      Pair          F          R2      p adjusted_p Significance
1 A4 vs F4 2.65489899 0.065303298 0.0877     0.5262         n.s.
2 A4 vs E4 1.34917874 0.034287342 0.2440     1.0000         n.s.
3 A4 vs G4 0.62343505 0.016141367 0.5042     1.0000         n.s.
4 F4 vs E4 1.27074982 0.032358685 0.2508     1.0000         n.s.
5 F4 vs G4 1.23149914 0.031390570 0.2751     1.0000         n.s.
6 E4 vs G4 0.06547484 0.001720059 0.9089     1.0000         n.s.


# Complementary post-hoc on dispersion (if PERMDISP is significant)

In [28]:
posthocPermdisp <- function(df, descriptors, factor, permutations = 9999) {
  
  printHeader(paste("POST-HOC ON DISPERSION:", toupper(factor)))
  
  standardized_data <- prepData(df, descriptors)
  dist_matrix <- calcDistanceMatrix(standardized_data, "euclidean")
  
  bd <- betadisper(dist_matrix, group = df[[factor]])
  
  # Tukey HSD
  distances <- data.frame(group = df[[factor]], distance = bd$distances)
  tukey_result <- TukeyHSD(aov(distance ~ group, data = distances))
  tukey_df <- as.data.frame(tukey_result$group)
  
  result_table <- data.frame(
    Pair = rownames(tukey_df),
    Δdispersion = tukey_df$diff,
    CI_lower = tukey_df$lwr,
    CI_upper = tukey_df$upr,
    adjusted_p = tukey_df$`p adj`
  )
  
  result_table <- colSignificance(result_table, "adjusted_p")
  
  print(result_table)
}

In [29]:
posthocPermdisp(
  df = data,
  descriptors = descriptors,
  factor = "method",
  permutations = 9999
)


POST-HOC ON DISPERSION: METHOD 

          Pair Δdispersion CI_lower  CI_upper  adjusted_p Significance
1 natural-bend   0.3901739 0.162435 0.6179128 0.001028649           **


# Sensitivity to the centroid window

`window_sensitivity.ipynb` recomputes the centroid under different window lengths and weightings and writes the values to `data/centroid_window_variants.csv` (so runs it before runs this).

In [30]:
variants <- read.csv('../data/centroid_window_variants.csv')

keys <- c("harp", "material", "method", "note", "rep")
window_data <- merge(
  data,
  variants[, c(keys, "A", "B", "D")],
  by = keys
)
cat(nrow(window_data), "recordings after the merge\n")

80 recordings after the merge


In [31]:
printHeader("OVERLAP CHECK")
for (v in c("A", "B", "D")) {
  cat(v,
      "max bend:", round(max(window_data[window_data$method == "bend", v])),
      " min natural:", round(min(window_data[window_data$method == "natural", v])), "\n")
}


OVERLAP CHECK 

A max bend: 2696  min natural: 4891 
B max bend: 3202  min natural: 5106 
D max bend: 1031  min natural: 1299 


In [32]:
sensitivityPermanova <- function(df, variant, permutations = 9999) {
  set.seed(20260601)

  df$centroid <- df[[variant]]

  dist_multi <- calcDistanceMatrix(prepData(df, descriptors), "euclidean")
  multi <- execPermanova(dist_multi, df, factors, permutations, by = "terms")

  dist_uni <- calcDistanceMatrix(scale(df$centroid))
  uni <- execPermanova(dist_uni, df, factors, permutations, by = "terms")

  summaryRow <- function(model, label) {
    data.frame(
      variant = variant,
      model = label,
      R2_method = model["method", "R2"],
      p_method = model["method", "Pr(>F)"],
      R2_material = model["material", "R2"],
      p_material = model["material", "Pr(>F)"]
    )
  }

  rbind(
    summaryRow(multi, "centroid + LAT"),
    summaryRow(uni, "centroid")
  )
}

In [33]:
printHeader("PERMANOVA BY CENTROID VARIANT")
sensitivity <- do.call(rbind, lapply(c("A", "B", "D"), function(v) {
  sensitivityPermanova(window_data, v, permutations = 9999)
}))
print(sensitivity, row.names = FALSE, digits = 4)


PERMANOVA BY CENTROID VARIANT 

 variant          model R2_method p_method R2_material p_material
       A centroid + LAT    0.6914    1e-04   0.0024543     0.3112
       A       centroid    0.8880    1e-04   0.0002926     0.0531
       B centroid + LAT    0.6879    1e-04   0.0026519     0.2891
       B       centroid    0.8810    1e-04   0.0006877     0.0089
       D centroid + LAT    0.5973    1e-04   0.0040455     0.1802
       D       centroid    0.6999    1e-04   0.0034750     0.0003
